# 3 · opencode — a coding agent on NRP

*Adapted from Daniel Diaz's CLARIPHY lesson [Agentic Workflows](https://training.nrp-nautilus.io/clariphy/3_agentic.html) ([nrp-nautilus/nrp-training](https://github.com/nrp-nautilus/nrp-training/tree/materials/clariphy)).*

Run cells with **Shift+Enter**. Cells share one persistent shell, so `export` and `cd` carry from cell to cell.

In [notebook 1](1_talk_to_a_model.ipynb) you **called** NRP's managed LLMs from Python, and in [notebook 2](2_agent_by_hand.ipynb) you ran the agent loop by hand. Now you will point an **agentic coding tool** — that same loop, with file and shell tools — at the same endpoint and have it plan, write, and run code autonomously on your behalf.

The key teaching point is **portability**: anything that speaks an OpenAI-compatible `base_url` works against NRP, so the agentic workflow you already use locally runs unchanged against NRP's managed inference — no API key handoff theater, no per-user billing.

> **Where to run what:** the install/config/setup steps below are real notebook cells (Shift+Enter). Launching `opencode` itself opens an interactive terminal UI, and the optional VS Code steps happen in an IDE — neither fits inside a notebook cell, so those are marked 🖥️ **Terminal step** / 💻 **IDE step** below and walked through in prose instead.
>
> Nothing here edits your shell startup files or any opencode configuration you already have: the exercise lives in a new `~/opencode-exercise` folder, plus one small file holding your token.

## Part 1: opencode

[`opencode`](https://opencode.ai) is an open-source terminal UI agentic coding assistant — similar in spirit to Claude Code or Cursor's CLI. It reads your project files, plans changes, edits code, and iterates.

### Install

If you already have opencode, this keeps your copy; otherwise it runs the official installer, which puts `opencode` in `~/.opencode/bin`:

In [ ]:
export PATH="$HOME/.opencode/bin:$PATH"
command -v opencode >/dev/null || curl -fsSL https://opencode.ai/install | bash
opencode --version


### Create the exercise folder

opencode scopes file writes to the nearest `.git` directory, not simply your
shell's current directory — without one, it can fall back to a much wider
default and write generated files somewhere you don't expect (a
[known opencode behavior](https://github.com/anomalyco/opencode/issues/15192),
not something specific to this training). So the exercise gets its own fresh
folder, turned into a git project:

In [ ]:
mkdir -p ~/opencode-exercise && cd ~/opencode-exercise
git init -q
pwd


### Configure NRP as the provider

opencode needs two things: where NRP's endpoint is, and your key. Notebook 1 already saved the key to
`~/.nrp-llm-token`, a private file in your home directory. Terminals in JupyterLab are **separate processes**
from this notebook, so `export`s here would not reach them; the file is how opencode will find the key (the
config below points at it). It sits outside the project, so it never ends up in the agent's working files or in
git. Check that it is there:

In [ ]:
if [ -s ~/.nrp-llm-token ]; then
    echo "Key found in ~/.nrp-llm-token"
    export OPENAI_API_KEY="$(cat ~/.nrp-llm-token)" OPENAI_API_BASE="https://ellm.nrp-nautilus.io/v1"
else
    echo "No key yet: run the key cell at the top of notebook 1 (1_talk_to_a_model.ipynb), then this cell again."
fi

Then write the NRP provider as a **project config** — an `opencode.json` inside the exercise folder. opencode merges it with any global config you have, with the project taking priority, so your own `~/.config/opencode` settings are left alone and this only applies inside `~/opencode-exercise`. (See the [full client-config reference](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/) for opencode, VS Code, Claude Code, and more.)

In [ ]:
cd ~/opencode-exercise
cat > opencode.json <<'JSON'
{
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "nrp": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "NRP LLM",
      "options": {
        "baseURL": "https://ellm.nrp-nautilus.io/v1",
        "apiKey": "{file:~/.nrp-llm-token}"
      },
      "models": {
        "deepseek-v4-flash": { "name": "DeepSeek V4 Flash" },
        "glm-5":             { "name": "GLM-5"             },
        "gpt-oss":           { "name": "GPT-OSS"           },
        "qwen3":             { "name": "Qwen3 397B"        }
      }
    }
  },
  "model": "nrp/deepseek-v4-flash"
}
JSON
cat opencode.json


`{file:~/.nrp-llm-token}` tells opencode to read your token from that file when it starts — so opencode works in **any** terminal, with nothing to export first — and with *your* token, even on the
training hub, where a shared `OPENAI_API_KEY` may already be set.

> **Switching models:** inside opencode, press **Ctrl+P** and select *Switch models* to change the active model mid-session. Try the same task with `deepseek-v4-flash` and `glm-5`: same tools, same files, different model — compare what each one does.

### Exercise: Build a CMS analysis helper

🖥️ **Terminal step** — `opencode` is an interactive terminal UI, so launch it from a JupyterLab terminal (**File → New → Terminal**), not this notebook:

```bash
cd ~/opencode-exercise
export PATH="$HOME/.opencode/bin:$PATH"
opencode
```

Getting `Forbidden` or `401` responses once inside opencode? It couldn't read a valid token — re-run the token cell above (with the placeholder replaced), then restart opencode.

The prompt is active as soon as opencode opens — just type your task and press Enter. (`/` opens the slash-command menu for things like `/models` or `/clear`, not the prompt itself.) Paste the following task:

```text
Write a Python script cms_nano_summary.py that uses the uproot library to open
a CMS NanoAOD ROOT file and print a summary of its contents.

The input is a real CMS NanoAOD file whose path is given as a command-line
argument. It has an "Events" TTree using the standard NanoAOD flat-branch
convention: collections appear as "<Collection>_<variable>" (Muon_pt,
Muon_eta, Jet_pt, ...), with an "n<Collection>" counter branch giving the
per-event multiplicity of each collection.

The script should:
- Group the branches by collection (all "Muon_*" together, all "Jet_*"
  together, and so on), listing anything that isn't part of a collection
  under "Event-level".
- For each branch print its name, type, and title/description if uproot
  exposes one.
- Print the total number of events at the end.
- Have a proper argparse interface and a top-level docstring.

Also write a requirements.txt pinning uproot>=5 and tabulate.
```

opencode will plan the implementation, write the files, and tell you how to run them. Back in this notebook (or a terminal), install the script's requirements into a **virtual environment** inside the project — its own private Python, so the install can't clash with (or be refused by) the system Python. Calling `.venv/bin/python` uses that environment directly, so there is nothing to activate:

In [ ]:
cd ~/opencode-exercise
python3 -m venv .venv
grep -qsxF '.venv/' .gitignore || echo '.venv/' >> .gitignore   # keep it out of git and the agent's searches
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python cms_nano_summary.py --help


### Test it on a real NanoAOD file

`--help` only proves the script parses. To actually exercise it you need a NanoAOD file, and there are two ways to get one.

In [ ]:
cd ~/opencode-exercise

# Pull the same NanoAOD file from NRP S3 — no proxy, no credentials (~20 MB).
[ -f nanoout_1.root ] || curl -fL -o nanoout_1.root \
  "https://s3-west.nrp-nautilus.io/transfer-bucket/QCD_Bin-Pt-15to7000_TuneCP5_13p6TeV_pythia8_nano.root" \
  || rm -f nanoout_1.root

ls -lh nanoout_1.root


In [ ]:
cd ~/opencode-exercise
.venv/bin/python cms_nano_summary.py nanoout_1.root | head -40


### Check the agent's work (NEW)

The script ran — but is its output **right**? Here the file itself is the planted truth: we know exactly which
columns it contains. This cell saves the script's output to `summary.txt` and compares it with the file: is every
column listed, and does each collection (Muon, Jet, boostedTau, …) get its own group?

In [ ]:
cd ~/opencode-exercise
.venv/bin/python cms_nano_summary.py nanoout_1.root > summary.txt 2>&1; echo "the script exited with status $?"
.venv/bin/python - <<'PY'
import re, uproot
keys = uproot.open("nanoout_1.root")["Events"].keys()
out = open("summary.txt").read()
missing = [k for k in keys if k not in out]
print(f"{len(keys)} columns in the file; {len(missing)} missing from the summary", missing[:5])
for group in ["Muon", "Jet", "Electron", "boostedTau"]:
    has_header = bool(re.search(rf"^\W*{group}\b", out, re.M))   # a line that starts with the group name
    print(f"{group:11s} has its own group:", "yes" if has_header else "NO - look at summary.txt")
PY

This is the real test of the agent's work: does the script actually survive contact with a NanoAOD file? Common ways a first attempt falls over — worth feeding straight back to opencode rather than fixing by hand:

- Treating every branch as flat when the jagged collection branches need `n<Collection>` to interpret.
- Crashing on branches with no title instead of printing a blank description.
- Assuming a fixed set of collections rather than discovering them from the file.

If it fails, paste the traceback into opencode and let it debug — watching an agent iterate on a real error is the point of the exercise.

> **Important:** if opencode generates a file named `uproot.py`, rename it — it would shadow the `uproot` package on import.

**Things to try (back in the opencode terminal):**
- Once the script is written, ask opencode to add a `--filter` argument that limits output to a specific collection (e.g., `--filter Muon`).
- Switch to `glm-5` and ask it to add unit tests with `pytest`.

## What opencode sends the model (NEW)

opencode is notebook 2's loop with more tools. Every turn it sends the model a `messages` list like the
ones you built by hand: its own long **system prompt** (how to behave, how to edit files), the **schemas
of its tools** (read, edit, bash, search, …), your request, and every tool call and result so far. Your
project files reach the model only through those tool results: it reads what it chooses to read.

That is why the context fills up in a long session (`/compact` summarises it), why the folder matters,
and why the same prompt behaves differently with `deepseek-v4-flash` and `glm-5`.

**Before Wednesday:** `opencode` should start in your JupyterLab terminal and answer with both models.
That is all the setup the hackathon needs.